In [1]:
import requests

API_KEY = "92dc6f0650ee8068f38dfd7eaa8b67c58a7ec1de25f5939ccd824af26e7eb479"
headers = {"X-API-Key": API_KEY}

url = "https://api.openaq.org/v3/locations"
params = {"limit": 1, "page": 1}

r = requests.get(url, headers=headers, params=params, timeout=30)
print("Status:", r.status_code)
print("First 300 chars:", r.text[:300])

r.raise_for_status()
data = r.json()
print("Meta:", data.get("meta"))
print("First result keys:", list(data["results"][0].keys()) if data.get("results") else "NO RESULTS")


StatementMeta(, 371258ba-e6be-4926-97e2-d94ab67c31e7, 3, Finished, Available, Finished)

Status: 200
First 300 chars: {"meta":{"name":"openaq-api","website":"/","page":1,"limit":1,"found":">1"},"results":[{"id":3,"name":"NMA - Nima","locality":null,"timezone":"Africa/Accra","country":{"id":152,"code":"GH","name":"Ghana"},"owner":{"id":4,"name":"Unknown Governmental Organization"},"provider":{"id":209,"name":"Dr. Ra
Meta: {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 1, 'found': '>1'}
First result keys: ['id', 'name', 'locality', 'timezone', 'country', 'owner', 'provider', 'isMobile', 'isMonitor', 'instruments', 'sensors', 'coordinates', 'licenses', 'bounds', 'distance', 'datetimeFirst', 'datetimeLast']


In [1]:
import requests
import time

API_KEY = "92dc6f0650ee8068f38dfd7eaa8b67c58a7ec1de25f5939ccd824af26e7eb479"
HEADERS = {"X-API-Key": API_KEY}

BASE = "https://api.openaq.org/v3"

# NYC bbox (minLon, minLat, maxLon, maxLat)
NYC_BBOX = "-74.2591,40.4774,-73.7004,40.9176"

def get_json(path, params=None):
    url = f"{BASE}{path}"
    r = requests.get(url, headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

# Pull some locations (keep it small first, then we can increase)
LIMIT = 100
PAGES = 3   # increase later if you want more
locations = []

for page in range(1, PAGES + 1):
    data = get_json("/locations", params={
        "country": "US",
        "bbox": NYC_BBOX,
        "limit": LIMIT,
        "page": page
    })
    locations.extend(data.get("results", []))
    print(f"Page {page}: got {len(data.get('results', []))} locations (total {len(locations)})")
    time.sleep(0.2)

# Extract sensor IDs from location objects
sensor_ids = set()
for loc in locations:
    sens = loc.get("sensors", [])
    for s in sens:
        if isinstance(s, dict) and "id" in s:
            sensor_ids.add(int(s["id"]))
        elif isinstance(s, int):
            sensor_ids.add(int(s))

sensor_ids = sorted(sensor_ids)
print("Unique sensors found:", len(sensor_ids))
print("First 20 sensor IDs:", sensor_ids[:20])


StatementMeta(, fb6e801c-6126-4e46-aea3-3cf7f9e2cee0, 3, Finished, Available, Finished)

Page 1: got 55 locations (total 55)
Page 2: got 0 locations (total 55)
Page 3: got 0 locations (total 55)
Unique sensors found: 194
First 20 sensor IDs: [671, 673, 674, 1097, 1098, 1099, 1102, 1103, 1106, 1121, 1128, 1143, 1145, 1146, 1147, 1152, 1522, 1523, 1534, 1535]


In [2]:
import json
import pandas as pd

MAX_SENSORS = 20
DATE_FROM = "2024-01-01T00:00:00Z"
DATE_TO   = "2024-12-31T23:59:59Z"

def pick_date(m):
    for k in ["date", "day", "period", "datetime"]:
        if k in m:
            v = m[k]
            if isinstance(v, dict):
                return v.get("utc") or v.get("local") or str(v)
            return str(v)
    return None

def pick_value(m):
    for k in ["value", "average", "mean", "median"]:
        if k in m:
            return m[k]
    return None

rows = []
chosen = sensor_ids[:MAX_SENSORS]
print("Downloading sensors:", len(chosen))

for i, sid in enumerate(chosen, start=1):
    # Get sensor metadata (helps us know parameter/unit if available)
    sensor_meta = get_json(f"/sensors/{sid}")
    sensor_result = (sensor_meta.get("results") or [sensor_meta])[0]  # handles both shapes

    parameter = sensor_result.get("parameter", {}).get("name") if isinstance(sensor_result.get("parameter"), dict) else sensor_result.get("parameter")
    unit = sensor_result.get("parameter", {}).get("units") if isinstance(sensor_result.get("parameter"), dict) else sensor_result.get("unit")

    # Get DAILY measurements
    page = 1
    while True:
        meas = get_json(f"/sensors/{sid}/measurements/daily", params={
            "datetime_from": DATE_FROM,
            "datetime_to": DATE_TO,
            "limit": 1000,
            "page": page
        })

        results = meas.get("results", [])
        if not results:
            break

        for m in results:
            rows.append({
                "sensor_id": sid,
                "parameter": parameter,
                "unit": unit,
                "date_utc": pick_date(m),
                "value": pick_value(m),
                "raw_json": json.dumps(m)
            })

        page += 1
        time.sleep(0.15)

    print(f"{i}/{len(chosen)} done (rows so far: {len(rows)})")

pdf = pd.DataFrame(rows)
print("Total rows downloaded:", len(pdf))
display(pdf.head(10))

df_bronze = spark.createDataFrame(pdf)

spark.sql("DROP TABLE IF EXISTS bronze_openaq_daily")
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_openaq_daily")
)

display(spark.table("bronze_openaq_daily").limit(10))
print("BRONZE OpenAQ rows:", spark.table("bronze_openaq_daily").count())


StatementMeta(, fb6e801c-6126-4e46-aea3-3cf7f9e2cee0, 4, Finished, Available, Finished)

1/20 done (rows so far: 366)
2/20 done (rows so far: 732)
3/20 done (rows so far: 732)
4/20 done (rows so far: 1099)
5/20 done (rows so far: 1461)
6/20 done (rows so far: 1461)
7/20 done (rows so far: 1817)
8/20 done (rows so far: 2183)
9/20 done (rows so far: 2550)
10/20 done (rows so far: 2550)
11/20 done (rows so far: 2915)
12/20 done (rows so far: 2915)
13/20 done (rows so far: 3279)
14/20 done (rows so far: 3646)
15/20 done (rows so far: 4013)
16/20 done (rows so far: 4379)
17/20 done (rows so far: 4379)
18/20 done (rows so far: 4379)
19/20 done (rows so far: 4736)
20/20 done (rows so far: 5103)
Total rows downloaded: 5103


SynapseWidget(Synapse.DataFrame, 4f2daef5-6c79-4d1f-b013-4726b1e4ef52)

SynapseWidget(Synapse.DataFrame, 4a88d66b-0751-4e97-a84f-f4bf2c4cf013)

BRONZE OpenAQ rows: 5103


In [3]:
from pyspark.sql import functions as F

bronze = spark.table("bronze_openaq_daily")

silver = (
    bronze
    # extract first YYYY-MM-DD from raw_json (works even if date is nested)
    .withColumn("date_str", F.regexp_extract(F.col("raw_json"), r"(\d{4}-\d{2}-\d{2})", 1))
    .withColumn("date", F.to_date("date_str"))
    .withColumn("value", F.col("value").cast("double"))
    .select(
        "sensor_id",
        "parameter",
        "unit",
        "date",
        "value",
        "raw_json"
    )
    .filter(F.col("date").isNotNull() & F.col("value").isNotNull())
)

spark.sql("DROP TABLE IF EXISTS silver_openaq_daily")
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_openaq_daily")
)

display(spark.table("silver_openaq_daily").limit(10))
print("SILVER OpenAQ rows:", spark.table("silver_openaq_daily").count())


StatementMeta(, fb6e801c-6126-4e46-aea3-3cf7f9e2cee0, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 2f9eebbb-fd5f-439e-8d59-d00f39e8b1ca)

SILVER OpenAQ rows: 5103


In [4]:
chk = spark.table("silver_openaq_daily")
chk.select(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
    F.sum(F.when(F.col("date").isNull(), 1).otherwise(0)).alias("null_dates"),
    F.sum(F.when(F.col("value").isNull(), 1).otherwise(0)).alias("null_values")
).show()

StatementMeta(, fb6e801c-6126-4e46-aea3-3cf7f9e2cee0, 6, Finished, Available, Finished)

+----------+----------+----------+-----------+
|  min_date|  max_date|null_dates|null_values|
+----------+----------+----------+-----------+
|2023-12-31|2024-12-31|         0|          0|
+----------+----------+----------+-----------+



In [5]:
from pyspark.sql import functions as F

silver = spark.table("silver_openaq_daily")

gold = (
    silver
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .groupBy("year", "month", "parameter", "unit")
    .agg(
        F.avg("value").alias("avg_value"),
        F.min("value").alias("min_value"),
        F.max("value").alias("max_value"),
        F.count("*").alias("days_count"),
        F.countDistinct("sensor_id").alias("sensor_count")
    )
    .orderBy("year", "month", "parameter")
)

spark.sql("DROP TABLE IF EXISTS gold_openaq_monthly")
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_openaq_monthly")
)

display(spark.table("gold_openaq_monthly").limit(50))
print("GOLD OpenAQ rows:", spark.table("gold_openaq_monthly").count())


StatementMeta(, fb6e801c-6126-4e46-aea3-3cf7f9e2cee0, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f60a1f2f-f78c-44f5-b36f-9aa8145e5ca5)

GOLD OpenAQ rows: 39
